# Notebook 5 — Mask Extraction & Boundary Vectorization


**Objectives**
- Convert a raster water mask into vector polygons (raster → vector)
- Compute area, perimeter, and compactness per polygon
- Filter noise polygons and flag geometric outliers
- Cross-check the SAR-derived water body against the MNDWI-derived water body

This notebook consumes outputs from Notebook 3 (`MNDWI`) and Notebook 4 (`naivasha_sar_water_mask.tif`) — run those first, or point the paths below at your own exports.

In [ ]:
import ee, geemap, numpy as np, rasterio
import geopandas as gpd
from shapely.geometry import shape

ee.Initialize(project="operating-rush-505912-s8")
AOI = ee.Geometry.Rectangle([36.20, -0.83, 36.45, -0.65])  # Lake Naivasha
CRS = "EPSG:32637"


## Raster → vector
Two equivalent paths: server-side with `ee.Image.reduceToVectors()` (scales to large areas, stays inside GEE), or local with `rasterio.features.shapes()` (once you already have the mask on disk from Notebook 4).

In [ ]:
# --- Server-side vectorization ---
sar_mask = ee.Image("projects/YOUR-GEE-PROJECT-ID/assets/naivasha_sar_water_mask")  # or re-load from Notebook 4

vectors_server = sar_mask.selfMask().reduceToVectors(
    geometry=AOI, scale=10, crs=CRS,
    geometryType="polygon", eightConnected=True,
    labelProperty="water", maxPixels=1e9
)
print("Server-side polygon count:", vectors_server.size().getInfo())

In [ ]:
# --- Local vectorization (from Notebook 4's exported .tif) ---
from rasterio.features import shapes

with rasterio.open("naivasha_sar_water_mask.tif") as src:
    mask_arr = src.read(1)
    transform = src.transform
    raster_crs = src.crs

polygons = [
    {"geometry": shape(geom), "value": val}
    for geom, val in shapes(mask_arr, mask=(mask_arr == 1), transform=transform)
]
sar_gdf = gpd.GeoDataFrame(polygons, crs=raster_crs).to_crs(CRS)
print("Local polygon count:", len(sar_gdf))

## Area, perimeter, compactness
Compactness (a.k.a. the Polsby–Popper score) compares a polygon's shape to a circle of the same area — the most efficient enclosure. **1.0** = perfect circle; values near 0 = long, thin, or highly irregular shapes. Useful for spotting SAR speckle-noise blobs (low compactness, tiny area) vs. the genuine lake body (large, moderately compact).

In [ ]:
sar_gdf["area_m2"] = sar_gdf.geometry.area
sar_gdf["perimeter_m"] = sar_gdf.geometry.length
sar_gdf["compactness"] = (4 * np.pi * sar_gdf["area_m2"]) / (sar_gdf["perimeter_m"] ** 2)

sar_gdf[["area_m2", "perimeter_m", "compactness"]].describe()

## Filtering small (noise) polygons
Anything under a sane minimum mapping unit is almost certainly leftover speckle rather than a real water feature. Tune the threshold to your sensor's ground sampling distance — for 10 m Sentinel-1, a handful of connected pixels (~400–900 m²) is a reasonable floor.

In [ ]:
MIN_AREA_M2 = 900  # ~9 pixels at 10 m resolution

sar_gdf_filtered = sar_gdf[sar_gdf["area_m2"] >= MIN_AREA_M2].reset_index(drop=True)
print(f"Polygons before filter: {len(sar_gdf)}  |  after filter: {len(sar_gdf_filtered)}")
print(f"Area removed as noise: {sar_gdf.loc[sar_gdf['area_m2'] < MIN_AREA_M2, 'area_m2'].sum():,.0f} m²")

## Outlier flagging
Beyond a hard area floor, flag polygons whose **compactness** is a statistical outlier relative to the rest of the set — these are shapes worth a human look even if they're not tiny (e.g. a thin arm of backscatter noise along a road, or a shadow artifact).

In [ ]:
q1, q3 = sar_gdf_filtered["compactness"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr

sar_gdf_filtered["outlier_flag"] = sar_gdf_filtered["compactness"] < lower_bound

print(f"Compactness IQR bounds: lower={lower_bound:.3f}")
print(f"Flagged as outliers: {sar_gdf_filtered['outlier_flag'].sum()} / {len(sar_gdf_filtered)}")

sar_gdf_filtered[sar_gdf_filtered["outlier_flag"]][["area_m2","perimeter_m","compactness"]]

## SAR-vs-MNDWI cross-check
The core validation step: does the SAR-derived water boundary agree with the optical (MNDWI-thresholded) water boundary from Notebook 3? Disagreement zones are exactly where you'd want a human to look before trusting either mask on its own — this reconciliation step is the direct antecedent of the production Riftwaters pipeline's QA stage.

In [ ]:
# Rebuild an MNDWI water mask (threshold at 0, the standard MNDWI water cutoff) and vectorize it
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(AOI).filterDate("2023-01-01", "2023-03-01")
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
      .median().clip(AOI))

mndwi = s2.normalizedDifference(["B3", "B11"])
mndwi_water = mndwi.gt(0).rename("water_mndwi")

mndwi_vectors = mndwi_water.selfMask().reduceToVectors(
    geometry=AOI, scale=10, crs=CRS, geometryType="polygon",
    eightConnected=True, labelProperty="water", maxPixels=1e9
)

geemap.ee_export_vector(mndwi_vectors, filename="naivasha_mndwi_water.geojson")
mndwi_gdf = gpd.read_file("naivasha_mndwi_water.geojson").to_crs(CRS)
print("MNDWI polygon count:", len(mndwi_gdf))

In [ ]:
# Dissolve each to a single multipolygon "lake body" for a clean overlap comparison
sar_dissolved = sar_gdf_filtered.geometry.union_all()
mndwi_dissolved = mndwi_gdf.geometry.union_all()

intersection = sar_dissolved.intersection(mndwi_dissolved)
union = sar_dissolved.union(mndwi_dissolved)

iou = intersection.area / union.area
sar_only_area = sar_dissolved.difference(mndwi_dissolved).area
mndwi_only_area = mndwi_dissolved.difference(sar_dissolved).area

print(f"IoU (agreement):        {iou:.3f}")
print(f"SAR-only area (m²):     {sar_only_area:,.0f}")
print(f"MNDWI-only area (m²):   {mndwi_only_area:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
gpd.GeoSeries([sar_dissolved], crs=CRS).boundary.plot(ax=ax, color="magenta", label="SAR boundary")
gpd.GeoSeries([mndwi_dissolved], crs=CRS).boundary.plot(ax=ax, color="blue", label="MNDWI boundary")
ax.legend(); ax.set_title(f"SAR vs MNDWI water boundary — IoU = {iou:.3f}")
plt.show()

**Reading the result:**
- High IoU (> ~0.9) on a stable lake like Naivasha in a cloud-free window is expected — both sensors should agree on the bulk of open water
- Persistent **SAR-only** slivers along the shore often indicate flooded vegetation or wet soil that SAR detects (via subsurface/volume scattering) but optical MNDWI misses because it's not spectrally "water"
- Persistent **MNDWI-only** slivers often indicate turbid or algae-covered water that scatters at radar wavelengths like land but still reads spectrally as water
- Neither sensor is "ground truth" — this cross-check quantifies *disagreement*, not error, and disagreement zones are what a human QA pass should prioritize

## Exercises Overview 

1. **Re-run the full Notebook 5 pipeline** on your Exercise 2.1/3.1/4.1 dry-season outputs
2. **Compare wet vs. dry season IoU** — does SAR/optical agreement improve or worsen in the dry season, and why?
3. **Tune `MIN_AREA_M2`** — plot polygon count retained vs. threshold value from 100–5,000 m² in steps of 200; pick and justify a final value for a "production" run
4. **Outlier review** — for every polygon flagged in the outlier step, manually classify it (noise / real feature / ambiguous) and report your precision (real+ambiguous / total flagged)
5. **Optional stretch:** repeat the whole notebook against a second Rift Valley lake of your choice (e.g. Lake Elementaita or Lake Bogoria) and compare compactness distributions between lakes

## Wrap-up — connection to the production Riftwaters pipeline
Everything built is a hand-run version of stages that run automatically, on a schedule, across every Rift Valley lake in the production **Riftwaters** pipeline:

| Workshop notebook | Production pipeline stage |
|---|---|
| Notebook 1–2 (acquisition) | Scheduled ingestion jobs, one per sensor, triggered on new scene availability |
| Notebook 3 (indices) | Server-side index computation, cached as pipeline assets |
| Notebook 4 (SAR processing) | Automated Otsu thresholding + cleanup, re-tuned per-region seasonally |
| Notebook 5 (vectorization + cross-check) | Automated QA gate — low-IoU dates get flagged for human review before publishing |

